In [2]:
import json
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
from constant import *
from GeminiModel import GeminiModel
load_dotenv()
from LlmSatdOutputLabelConverter import LlmSatdOutputLabelConverter
from Model import Model
from ChatGpt4Model import ChatGpt4Model

/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
output_label_converter = LlmSatdOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)
def parse_batch_output(model:Model, file, model_suffix):
    df = pd.concat([detect_train_df, detect_test_df], ignore_index=True) if model.task_type == 'detect' else pd.concat([classify_train_df, classify_test_df], ignore_index=True)
    batch_dataset = Dataset.from_pandas(df)
    with open(file, 'r', encoding='utf-8') as json_output_file:
                            for line in json_output_file:
                                output = json.loads((line.strip()))
                                sample_id = int(output['key'])
                                raw_predicted_label = output['response']['candidates'][-1]['content']['parts'][-1]['text']
                                predicted_label = model.output_label_converter.convert_label(raw_predicted_label)
                                model.append_into_merged_file(model_suffix, batch_dataset, sample_id, predicted_label,raw_predicted_label)


def get_completed_status_set(model_uri):
    if 'gemini' in model_uri:
        return {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED', 'JOB_STATE_EXPIRED'}
    elif 'gpt' in model_uri:
        return {'failed', 'finalizing', 'completed', 'expired', 'cancelling','cancelled' }
    else:
        return {}


In [ ]:
# parse_batch_output(GeminiModel('detect', 'models/gemini-2.0-flash', output_label_converter, True), '/home/cs/grad/islams32/dev/project/academic/technical-debt/cache/output/batch/output/detect_gemini-2.0-flash-2-shot.jsonl', '2-shot')
JOB_FILE = f'{os.getenv("CACHE_DIRECTORY")}/batch/job.csv'

In [9]:
import time



tryAgain = True
while tryAgain:
    tryAgain = False
    job_df = pd.read_csv(JOB_FILE, dtype={"status": "string"})
    for idx, row in job_df.iterrows():
        model_uri = row["model_uri"]
        completed_states = get_completed_status_set(model_uri)
        if row["status"] not in completed_states:
            tryAgain = True
            if  'gemini' in  model_uri:
                model = GeminiModel(row['task_type'], model_uri, output_label_converter, True)
            elif 'gpt' in model_uri:
                model = ChatGpt4Model(row['task_type'], model_uri, output_label_converter, True)
            batch_job = model.model.batches.get(name=row["job_id"])
            job_name = row['job_name']
            if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
                    if batch_job.dest and batch_job.dest.file_name:
                        # Results are in a file
                        result_file_name = batch_job.dest.file_name
                        print(f"Results are in file: {result_file_name}")

                        print("Downloading result file content...")
                        file_content = gemini_model.model.files.download(file=result_file_name)
                        output_file_name = row['input_file'].replace('/input/', '/output/')
                        os.makedirs(os.path.dirname(output_file_name), exist_ok=True)
                        with open(output_file_name, 'w') as output_file:
                            output_file.write(file_content.decode('utf-8'))
                        model_suffix = job_name[job_name.rfind('-',0, len(job_name) - len('-shot')) + 1:]
                        parse_batch_output(model, output_file_name, model_suffix)
            else:
                print(f"Job status detail {batch_job}")
            job_df.at[idx, "status"] = batch_job.state.name
    job_df.to_csv(gemini_model.get_batch_job_file_name(), index=False)
    if tryAgain:
        print('trying..')
        time.sleep(30)


Results are in file: files/batch-hopwucldckeqnea1wd2e41odls4ab5hn7jy1
Job status detail name='batches/gjtfro0n6uqcg6ajyxqfpw9whmlqyzb8d2zp' display_name='detect_gemini-2.0-flash-2-shot' state=<JobState.JOB_STATE_PENDING: 'JOB_STATE_PENDING'> error=None create_time=datetime.datetime(2025, 9, 16, 21, 40, 42, 141094, tzinfo=TzInfo(UTC)) start_time=None end_time=None update_time=datetime.datetime(2025, 9, 16, 21, 40, 42, 141094, tzinfo=TzInfo(UTC)) model='models/gemini-2.0-flash' src=None dest=None
trying..
Job status detail name='batches/gjtfro0n6uqcg6ajyxqfpw9whmlqyzb8d2zp' display_name='detect_gemini-2.0-flash-2-shot' state=<JobState.JOB_STATE_PENDING: 'JOB_STATE_PENDING'> error=None create_time=datetime.datetime(2025, 9, 16, 21, 40, 42, 141094, tzinfo=TzInfo(UTC)) start_time=None end_time=None update_time=datetime.datetime(2025, 9, 16, 21, 40, 42, 141094, tzinfo=TzInfo(UTC)) model='models/gemini-2.0-flash' src=None dest=None
trying..
Results are in file: files/batch-gjtfro0n6uqcg6ajyxq